In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from pathlib import Path
from IPython.display import display

print("Dashboard libraries successfully loaded!")

Dashboard libraries successfully loaded!


In [2]:
# Define project paths
PROJECT_DIR = Path.cwd().parent
DATA_DIR = PROJECT_DIR / "data" / "processed"

# Load cleaned datasets
customers = pd.read_csv(DATA_DIR / "customers_cleaned.csv")
orders = pd.read_csv(DATA_DIR / "orders_cleaned.csv")
order_items = pd.read_csv(DATA_DIR / "order_items_cleaned.csv")
payments = pd.read_csv(DATA_DIR / "payments_cleaned.csv")
reviews = pd.read_csv(DATA_DIR / "reviews_cleaned.csv")
products = pd.read_csv(DATA_DIR / "products_cleaned.csv")
sellers = pd.read_csv(DATA_DIR / "sellers_cleaned.csv")
category_translation = pd.read_csv(DATA_DIR / "category_translation_cleaned.csv")

print("All cleaned datasets loaded successfully!")

All cleaned datasets loaded successfully!


In [3]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows | {df.shape[1]} columns")

Customers: 99,441 rows | 5 columns
Orders: 99,441 rows | 8 columns
Order Items: 112,650 rows | 7 columns
Payments: 103,886 rows | 5 columns
Reviews: 99,224 rows | 7 columns
Products: 32,951 rows | 9 columns
Sellers: 3,095 rows | 4 columns
Category Translation: 71 rows | 2 columns


In [4]:
# Calculate main business KPIs

total_sales = order_items["price"].sum()
total_orders = orders["order_id"].nunique()
total_items_sold = len(order_items)
total_customers = customers["customer_unique_id"].nunique()
average_order_value = total_sales / total_orders

# Display KPI results
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Sales",
        "Total Orders",
        "Total Items Sold",
        "Total Customers",
        "Average Order Value"
    ],
    "Value": [
        f"R$ {total_sales:,.2f}",
        f"{total_orders:,}",
        f"{total_items_sold:,}",
        f"{total_customers:,}",
        f"R$ {average_order_value:,.2f}"
    ]
})

display(kpi_summary)

,KPI,Value
0,Total Sales,"R$ 13,591,643.70"
1,Total Orders,"99,441"
2,Total Items Sold,"112,650"
3,Total Customers,"96,096"
4,Average Order Value,R$ 136.68


In [5]:
# Calculate main KPIs

total_sales = order_items["price"].sum()
total_orders = orders["order_id"].nunique()
total_items_sold = len(order_items)
total_customers = customers["customer_unique_id"].nunique()
average_order_value = total_sales / total_orders

print(f"Total Sales: R$ {total_sales:,.2f}")
print(f"Total Orders: {total_orders:,}")
print(f"Total Items Sold: {total_items_sold:,}")
print(f"Total Customers: {total_customers:,}")
print(f"Average Order Value: R$ {average_order_value:,.2f}")

Total Sales: R$ 13,591,643.70
Total Orders: 99,441
Total Items Sold: 112,650
Total Customers: 96,096
Average Order Value: R$ 136.68


In [6]:
from plotly.subplots import make_subplots

# Create KPI card layout
fig_kpi = make_subplots(
    rows=1,
    cols=5,
    specs=[[{"type": "indicator"}] * 5],
    horizontal_spacing=0.04
)

kpi_data = [
    ("TOTAL SALES", total_sales, "R$ ", ",.2f", "#2563EB"),
    ("TOTAL ORDERS", total_orders, None, ",.0f", "#0F766E"),
    ("ITEMS SOLD", total_items_sold, None, ",.0f", "#7C3AED"),
    ("CUSTOMERS", total_customers, None, ",.0f", "#D97706"),
    ("AVERAGE ORDER VALUE", average_order_value, "R$ ", ",.2f", "#0891B2")
]

for col, (label, value, prefix, fmt, color) in enumerate(kpi_data, start=1):
    fig_kpi.add_trace(
        go.Indicator(
            mode="number",
            value=value,
            number={
                "prefix": prefix or "",
                "valueformat": fmt,
                "font": {"size": 25, "color": color}
            },
            title={
                "text": f"<b>{label}</b>",
                "font": {"size": 12, "color": "#64748B"}
            },
            domain={"row": 0, "column": col - 1}
        ),
        row=1,
        col=col
    )

fig_kpi.update_layout(
    template="plotly_white",
    height=180,
    margin=dict(l=15, r=15, t=35, b=15),
    paper_bgcolor="#F1F5F9",
    font_family="Arial"
)

fig_kpi.show()

In [7]:
# Create improved KPI Cards

kpi_data = [
    {
        "label": "TOTAL SALES",
        "value": total_sales / 1_000_000,
        "prefix": "R$ ",
        "suffix": "M",
        "format": ".2f",
        "color": "#2563EB"
    },
    {
        "label": "TOTAL ORDERS",
        "value": total_orders,
        "prefix": "",
        "suffix": "",
        "format": ",.0f",
        "color": "#0F766E"
    },
    {
        "label": "ITEMS SOLD",
        "value": total_items_sold,
        "prefix": "",
        "suffix": "",
        "format": ",.0f",
        "color": "#7C3AED"
    },
    {
        "label": "CUSTOMERS",
        "value": total_customers,
        "prefix": "",
        "suffix": "",
        "format": ",.0f",
        "color": "#D97706"
    },
    {
        "label": "AVERAGE ORDER VALUE",
        "value": average_order_value,
        "prefix": "R$ ",
        "suffix": "",
        "format": ",.2f",
        "color": "#0891B2"
    }
]

# Initialize figure
fig_kpi = go.Figure()

# Add individual KPI cards
for i, kpi in enumerate(kpi_data):

    x0 = 0.025 + i * 0.195
    x1 = x0 + 0.17

    # Card background
    fig_kpi.add_shape(
        type="rect",
        xref="paper",
        yref="paper",
        x0=x0,
        x1=x1,
        y0=0.12,
        y1=0.88,
        fillcolor="#FFFFFF",
        line=dict(color="#E2E8F0", width=1),
        layer="below"
    )

    # KPI value and title
    fig_kpi.add_trace(
        go.Indicator(
            mode="number",
            value=kpi["value"],
            title={
                "text": f"<b>{kpi['label']}</b>",
                "font": {
                    "size": 12,
                    "color": "#64748B"
                }
            },
            number={
                "prefix": kpi["prefix"],
                "suffix": kpi["suffix"],
                "valueformat": kpi["format"],
                "font": {
                    "size": 21,
                    "color": kpi["color"]
                }
            },
            domain={
                "x": [x0 + 0.005, x1 - 0.005],
                "y": [0.25, 0.75]
            }
        )
    )

# Dashboard styling
fig_kpi.update_layout(
    title=None,
    width=1400,
    height=250,
    template="plotly_white",
    paper_bgcolor="#F1F5F9",
    plot_bgcolor="#F1F5F9",
    margin=dict(l=10, r=10, t=15, b=15),
    font_family="Arial"
)

fig_kpi.show()

In [9]:
print(orders.columns.tolist())

['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']


In [10]:
# Prepare monthly sales data

orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

sales_data = order_items.merge(
    orders[["order_id", "order_purchase_timestamp"]],
    on="order_id",
    how="left"
)

sales_data["month"] = (
    sales_data["order_purchase_timestamp"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

monthly_sales = (
    sales_data
    .groupby("month", as_index=False)
    .agg(total_sales=("price", "sum"))
)

# Exclude incomplete September and October 2018
monthly_sales = monthly_sales[
    ~monthly_sales["month"].isin(
        pd.to_datetime(["2018-09-01", "2018-10-01"])
    )
]

display(monthly_sales.head())

,month,total_sales
0,2016-09-01,267.36
1,2016-10-01,49507.66
2,2016-12-01,10.90
3,2017-01-01,120312.87
4,2017-02-01,247303.02


In [12]:
# Create Monthly Sales Trend chart

fig_monthly = px.line(
    monthly_sales,
    x="month",
    y="total_sales",
    markers=True,
    labels={
        "month": "Month",
        "total_sales": "Sales Value (R$)"
    }
)

fig_monthly.update_traces(
    line=dict(color="#2563EB", width=3),
    marker=dict(size=8, color="#0F766E"),
    fill="tozeroy",
    fillcolor="rgba(37, 99, 235, 0.08)"
)

fig_monthly.update_layout(
    title=dict(
        text="Monthly Sales Trend",
        font=dict(size=22, color="#17365D")
    ),
    template="plotly_white",
    width=1100,
    height=450,
    xaxis_title="Month",
    yaxis_title="Sales Value (R$)",
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF",
    hovermode="x unified"
)

fig_monthly.show()

In [14]:
OUTPUT_DIR = PROJECT_DIR / "reports" / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Folder penyimpanan berhasil disiapkan:")
print(OUTPUT_DIR)

Folder penyimpanan berhasil disiapkan:
d:\PORTOFOLIO\ecommerce-sales-analytics\reports\figures


In [15]:
fig_monthly.write_html(
    OUTPUT_DIR / "monthly_sales_trend.html"
)

print("Monthly Sales Trend successfully exported!")

Monthly Sales Trend successfully exported!


In [16]:
# Export Executive KPI Cards

fig_kpi.write_html(
    OUTPUT_DIR / "executive_kpi_cards.html"
)

print("Executive KPI Cards successfully exported!")

Executive KPI Cards successfully exported!


In [17]:
# Merge order items with products
product_sales = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

# Translate category names into English
product_sales = product_sales.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

# Aggregate sales by category
category_sales = (
    product_sales
    .groupby("product_category_name_english", as_index=False)
    .agg(total_sales=("price", "sum"))
    .sort_values("total_sales", ascending=False)
    .head(10)
)

display(category_sales)

,product_category_name_english,total_sales
43,health_beauty,1258681.34
70,watches_gifts,1205005.68
7,bed_bath_table,1036988.68
65,sports_leisure,988048.97
15,computers_accessories,911954.32
39,furniture_decor,729762.49
20,cool_stuff,635290.85
49,housewares,632248.66
5,auto,592720.11
42,garden_tools,485256.46


In [18]:
fig_category = px.bar(
    category_sales.sort_values("total_sales"),
    x="total_sales",
    y="product_category_name_english",
    orientation="h",
    text="total_sales",
    title="Top 10 Product Categories by Sales",
    labels={
        "total_sales": "Sales Value (R$)",
        "product_category_name_english": "Product Category"
    }
)

fig_category.update_traces(
    marker_color="#2563EB",
    texttemplate="R$ %{x:,.0f}",
    textposition="outside"
)

fig_category.update_layout(
    template="plotly_white",
    width=1100,
    height=550,
    title_font=dict(size=22, color="#17365D"),
    xaxis_title="Sales Value (R$)",
    yaxis_title="Product Category",
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF"
)

fig_category.show()

In [19]:
fig_category.write_html(
    OUTPUT_DIR / "top_10_product_categories.html"
)

print("Product Category Chart successfully exported!")

Product Category Chart successfully exported!


In [20]:
# Aggregate payment data by payment method
payment_analysis = (
    payments
    .groupby("payment_type", as_index=False)
    .agg(
        total_transactions=("payment_type", "count"),
        total_payment_value=("payment_value", "sum")
    )
    .sort_values("total_payment_value", ascending=False)
)

display(payment_analysis)

,payment_type,total_transactions,total_payment_value
1,credit_card,76795,12542084.19
0,boleto,19784,2869361.27
4,voucher,5775,379436.87
2,debit_card,1529,217989.79
3,not_defined,3,0.00


In [21]:
fig_payment = px.bar(
    payment_analysis,
    x="payment_type",
    y="total_payment_value",
    text="total_payment_value",
    title="Payment Methods by Total Transaction Value",
    labels={
        "payment_type": "Payment Method",
        "total_payment_value": "Total Payment Value (R$)"
    }
)

fig_payment.update_traces(
    marker_color="#0F766E",
    texttemplate="R$ %{y:,.0f}",
    textposition="outside"
)

fig_payment.update_layout(
    template="plotly_white",
    width=1100,
    height=500,
    title_font=dict(size=22, color="#17365D"),
    xaxis_title="Payment Method",
    yaxis_title="Total Payment Value (R$)",
    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF"
)

fig_payment.show()

In [22]:
fig_payment.write_html(
    OUTPUT_DIR / "payment_methods_analysis.html"
)

print("Payment Methods Chart successfully exported!")

Payment Methods Chart successfully exported!


In [23]:
# Aggregate order status
order_status = (
    orders["order_status"]
    .value_counts()
    .reset_index()
)

order_status.columns = ["order_status", "total_orders"]

display(order_status)

,order_status,total_orders
0,delivered,96478
1,shipped,1107
2,canceled,625
3,unavailable,609
4,invoiced,314
5,processing,301
6,created,5
7,approved,2


In [24]:
fig_status = px.bar(
    order_status.sort_values("total_orders"),
    x="total_orders",
    y="order_status",
    orientation="h",
    text="total_orders",
    title="Order Status Distribution",
    labels={
        "total_orders": "Number of Orders",
        "order_status": "Order Status"
    }
)

fig_status.update_traces(
    marker_color="#6366F1",
    texttemplate="%{x:,.0f}",
    textposition="outside"
)

fig_status.update_layout(
    template="plotly_white",
    width=1100,
    height=500,
    title_font=dict(size=22, color="#17365D"),
    xaxis_title="Number of Orders",
    yaxis_title="Order Status"
)

fig_status.show()

fig_status.write_html(
    OUTPUT_DIR / "order_status_distribution.html"
)

In [25]:
# Customer distribution by state
customer_state = (
    customers.groupby("customer_state", as_index=False)
    .agg(total_customers=("customer_unique_id", "nunique"))
    .sort_values("total_customers", ascending=False)
)

# Customer purchase frequency
customer_orders = orders.merge(
    customers[["customer_id", "customer_unique_id"]],
    on="customer_id",
    how="left"
)

purchase_frequency = (
    customer_orders.groupby("customer_unique_id")
    .agg(total_orders=("order_id", "nunique"))
    .reset_index()
)

purchase_frequency["customer_type"] = np.where(
    purchase_frequency["total_orders"] > 1,
    "Repeat Customer",
    "One-Time Customer"
)

customer_type = (
    purchase_frequency["customer_type"]
    .value_counts()
    .reset_index()
)

customer_type.columns = ["customer_type", "total_customers"]

display(customer_state.head(10))
display(customer_type)

,customer_state,total_customers
25,SP,40302
18,RJ,12384
10,MG,11259
22,RS,5277
17,PR,4882
23,SC,3534
4,BA,3277
6,DF,2075
7,ES,1964
8,GO,1952


,customer_type,total_customers
0,One-Time Customer,93099
1,Repeat Customer,2997


In [26]:
fig_customer_state = px.bar(
    customer_state.head(10).sort_values("total_customers"),
    x="total_customers",
    y="customer_state",
    orientation="h",
    text="total_customers",
    title="Top 10 Customer States",
    labels={
        "total_customers": "Number of Customers",
        "customer_state": "Customer State"
    }
)

fig_customer_state.update_traces(
    marker_color="#0891B2",
    textposition="outside"
)

fig_customer_state.update_layout(
    template="plotly_white",
    width=1100,
    height=500,
    title_font=dict(size=22, color="#17365D")
)

fig_customer_state.show()

fig_customer_state.write_html(
    OUTPUT_DIR / "customer_distribution_by_state.html"
)

In [27]:
fig_customer_type = px.pie(
    customer_type,
    names="customer_type",
    values="total_customers",
    title="Customer Purchase Frequency",
    hole=0.45
)

fig_customer_type.update_layout(
    template="plotly_white",
    width=900,
    height=500,
    title_font=dict(size=22, color="#17365D")
)

fig_customer_type.show()

fig_customer_type.write_html(
    OUTPUT_DIR / "customer_purchase_frequency.html"
)

In [28]:
# Convert delivery date columns
delivery_columns = [
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in delivery_columns:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Select delivered orders with valid delivery dates
delivery_analysis = orders[
    orders["order_status"].eq("delivered")
].copy()

delivery_analysis = delivery_analysis.dropna(
    subset=delivery_columns
)

# Calculate actual and estimated delivery duration
delivery_analysis["actual_delivery_days"] = (
    delivery_analysis["order_delivered_customer_date"]
    - delivery_analysis["order_purchase_timestamp"]
)

delivery_analysis["actual_delivery_days"] = (
    delivery_analysis["actual_delivery_days"].dt.total_seconds() / 86400
)

delivery_analysis["estimated_delivery_days"] = (
    delivery_analysis["order_estimated_delivery_date"]
    - delivery_analysis["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Classify delivery performance
delivery_analysis["delivery_status"] = np.where(
    delivery_analysis["order_delivered_customer_date"]
    <= delivery_analysis["order_estimated_delivery_date"],
    "On-Time",
    "Late"
)

delivery_summary = (
    delivery_analysis["delivery_status"]
    .value_counts()
    .reset_index()
)

delivery_summary.columns = ["delivery_status", "total_orders"]

display(delivery_summary)

print(
    "Average Actual Delivery:",
    round(delivery_analysis["actual_delivery_days"].mean(), 2),
    "days"
)

print(
    "Average Estimated Delivery:",
    round(delivery_analysis["estimated_delivery_days"].mean(), 2),
    "days"
)

,delivery_status,total_orders
0,On-Time,88644
1,Late,7826


Average Actual Delivery: 12.56 days
Average Estimated Delivery: 23.74 days


In [29]:
# Merge reviews with delivery status
review_analysis = reviews.merge(
    delivery_analysis[["order_id", "delivery_status"]],
    on="order_id",
    how="inner"
)

# Calculate average review score
review_summary = (
    review_analysis.groupby("delivery_status", as_index=False)
    .agg(
        total_reviews=("review_score", "count"),
        average_review_score=("review_score", "mean")
    )
)

review_summary["average_review_score"] = (
    review_summary["average_review_score"].round(2)
)

display(review_summary)

,delivery_status,total_reviews,average_review_score
0,Late,7700,2.57
1,On-Time,88653,4.29


In [30]:
fig_review = px.bar(
    review_summary,
    x="delivery_status",
    y="average_review_score",
    text="average_review_score",
    title="Average Review Score by Delivery Performance",
    labels={
        "delivery_status": "Delivery Performance",
        "average_review_score": "Average Review Score"
    },
    range_y=[0, 5]
)

fig_review.update_traces(
    marker_color="#D97706",
    texttemplate="%{y:.2f}",
    textposition="outside"
)

fig_review.update_layout(
    template="plotly_white",
    width=900,
    height=500,
    title_font=dict(size=22, color="#17365D"),
    yaxis_title="Average Review Score"
)

fig_review.show()

fig_review.write_html(
    OUTPUT_DIR / "review_score_by_delivery.html"
)

In [31]:
# Merge order items with seller information
seller_sales = order_items.merge(
    sellers[["seller_id", "seller_state"]],
    on="seller_id",
    how="left"
)

# Aggregate seller performance
seller_performance = (
    seller_sales.groupby(["seller_id", "seller_state"], as_index=False)
    .agg(
        total_orders=("order_id", "nunique"),
        total_items=("order_id", "count"),
        total_sales=("price", "sum")
    )
    .sort_values("total_sales", ascending=False)
)

top_sellers = seller_performance.head(10).copy()

display(top_sellers)

,seller_id,seller_state,total_orders,total_items,total_sales
857,4869f7a5dfa277a7dca6462dcf3b52b2,SP,1132,1156,229472.63
1013,53243585a1d6dc2643021fd1853d8905,BA,358,410,222776.05
881,4a3ca9315b744ce9f8e9374361493884,SP,1806,1987,200472.92
3024,fa1c13f2614d7b5c4749cbc52fecda94,SP,585,586,194042.03
1535,7c67e1448b00f6e969d365cea6b010ab,SP,982,1364,187923.89
1560,7e93a43ef30c4f03f38b393420bc753a,SP,336,340,176431.87
2643,da8622b14eb17ae2831f4ac5b9dab84a,SP,1314,1551,160236.57
1505,7a67c85e85bb2ce8582c35f2203ad736,SP,1160,1171,141745.53
192,1025f0e2d44d7041d6cf58b6550e0bfa,SP,915,1428,138968.55
1824,955fee9216a65b617aa5c0531780ce60,SP,1287,1499,135171.70


In [32]:
fig_seller = px.bar(
    top_sellers.sort_values("total_sales"),
    x="total_sales",
    y="seller_id",
    orientation="h",
    text="total_sales",
    title="Top 10 Sellers by Sales Value",
    labels={
        "total_sales": "Sales Value (R$)",
        "seller_id": "Seller ID"
    }
)

fig_seller.update_traces(
    marker_color="#7C3AED",
    texttemplate="R$ %{x:,.0f}",
    textposition="outside"
)

fig_seller.update_layout(
    template="plotly_white",
    width=1100,
    height=600,
    title_font=dict(size=22, color="#17365D")
)

fig_seller.show()

fig_seller.write_html(
    OUTPUT_DIR / "top_10_sellers.html"
)

In [33]:
seller_state_analysis = (
    seller_sales.groupby("seller_state", as_index=False)
    .agg(
        total_sellers=("seller_id", "nunique"),
        total_orders=("order_id", "nunique"),
        total_sales=("price", "sum")
    )
    .sort_values("total_sales", ascending=False)
)

display(seller_state_analysis.head(10))

,seller_state,total_sellers,total_orders,total_sales
22,SP,1849,70188,8753396.21
15,PR,349,7673,1261887.21
8,MG,244,7930,1011564.74
16,RJ,171,4353,843984.22
20,SC,190,3667,632426.07
19,RS,129,1989,378559.54
2,BA,19,569,285561.56
4,DF,30,824,97749.48
13,PE,9,406,91493.85
6,GO,40,463,66399.21


In [34]:
fig_seller_state = px.bar(
    seller_state_analysis.head(10).sort_values("total_sales"),
    x="total_sales",
    y="seller_state",
    orientation="h",
    text="total_sales",
    title="Top 10 Seller States by Sales Value",
    labels={
        "total_sales": "Sales Value (R$)",
        "seller_state": "Seller State"
    }
)

fig_seller_state.update_traces(
    marker_color="#0F766E",
    texttemplate="R$ %{x:,.0f}",
    textposition="outside"
)

fig_seller_state.update_layout(
    template="plotly_white",
    width=1100,
    height=550,
    title_font=dict(size=22, color="#17365D")
)

fig_seller_state.show()

fig_seller_state.write_html(
    OUTPUT_DIR / "seller_performance_by_state.html"
)

In [35]:
# List all exported dashboard visualizations
exported_files = sorted(OUTPUT_DIR.glob("*.html"))

print("TOTAL EXPORTED VISUALIZATIONS:", len(exported_files))
print("\nEXPORTED FILES:")

for file in exported_files:
    print("-", file.name)

TOTAL EXPORTED VISUALIZATIONS: 10

EXPORTED FILES:
- customer_distribution_by_state.html
- customer_purchase_frequency.html
- executive_kpi_cards.html
- monthly_sales_trend.html
- order_status_distribution.html
- payment_methods_analysis.html
- review_score_by_delivery.html
- seller_performance_by_state.html
- top_10_product_categories.html
- top_10_sellers.html


In [37]:
fig_delivery = px.pie(
    delivery_summary,
    names="delivery_status",
    values="total_orders",
    title="On-Time vs Late Delivery Performance",
    hole=0.45
)

fig_delivery.update_layout(
    template="plotly_white",
    width=900,
    height=500,
    title_font=dict(size=22, color="#17365D")
)

fig_delivery.show()

In [38]:
fig_delivery.write_html(
    OUTPUT_DIR / "delivery_performance.html"
)

print("Delivery Performance successfully exported!")

Delivery Performance successfully exported!


In [39]:
exported_files = sorted(OUTPUT_DIR.glob("*.html"))

print("TOTAL EXPORTED VISUALIZATIONS:", len(exported_files))

for file in exported_files:
    print("-", file.name)

TOTAL EXPORTED VISUALIZATIONS: 11
- customer_distribution_by_state.html
- customer_purchase_frequency.html
- delivery_performance.html
- executive_kpi_cards.html
- monthly_sales_trend.html
- order_status_distribution.html
- payment_methods_analysis.html
- review_score_by_delivery.html
- seller_performance_by_state.html
- top_10_product_categories.html
- top_10_sellers.html
